##

### 분기별 데이터 확인

In [44]:
import sqlite3
import pandas as pd
import numpy as np

# 각 분기 DB 설정 (테이블명 오타 포함 그대로 사용)
DB_CONFIGS = [
    {'db': '../data/quarter1_data.db', 'table': '1q_financial',    'quarter': 1},
    {'db': '../data/quarter2_data.db', 'table': '2q_finanical',    'quarter': 2},  # 원본 오타
    {'db': '../data/quarter3_data.db', 'table': '3q_finanical',    'quarter': 3},  # 원본 오타
    {'db': '../data/quarter4_data.db', 'table': 'annual_financial', 'quarter': 4},
]

# 각 분기 DF 개별 저장
dfs = {}
for cfg in DB_CONFIGS:
    conn = sqlite3.connect(cfg['db'])
    df = pd.read_sql_query(f"SELECT * FROM \"{cfg['table']}\"", conn)
    conn.close()
    dfs[cfg['quarter']] = df
    print(f"Q{cfg['quarter']} [{cfg['table']}]: {len(df):,}행, {df.shape[1]}열")

df_q1 = dfs[1]
df_q2 = dfs[2]
df_q3 = dfs[3]
df_q4 = dfs[4]

df_q1.to_csv('../data/quarter1_finance.csv', index=False)
df_q2.to_csv('../data/quarter2_finance.csv', index=False)
df_q3.to_csv('../data/quarter3_finance.csv', index=False)
df_q4.to_csv('../data/quarter4_finance.csv', index=False)

Q1 [1q_financial]: 21,432행, 35열
Q2 [2q_finanical]: 21,720행, 34열
Q3 [3q_finanical]: 22,024행, 35열
Q4 [annual_financial]: 21,994행, 34열


In [21]:
# net_income 제외 후 분기 컬럼 추가, concat
quarter_df = pd.concat(
    [df.drop(columns=['net_income'], errors='ignore').assign(quarter=q)
     for q, df in [(1, df_q1), (2, df_q2), (3, df_q3), (4, df_q4)]],
    ignore_index=True,
    sort=False,
)
quarter_df = quarter_df.sort_values(['stock_code', 'year', 'quarter']).reset_index(drop=True)

print(f"합산: {len(quarter_df):,}행, {quarter_df.shape[1]}열")
print(f"\n[quarter 분포]")
print(quarter_df['quarter'].value_counts().sort_index())

# 정렬 검증
is_sorted = (
    quarter_df.groupby('stock_code')
    .apply(lambda g: g[['year', 'quarter']].reset_index(drop=True)
                       .equals(g[['year', 'quarter']].sort_values(['year', 'quarter']).reset_index(drop=True)),
           include_groups=False)
    .all()
)
print(f"\n(stock_code, year, quarter) 정렬 검증: {is_sorted}")

합산: 87,170행, 35열

[quarter 분포]
quarter
1    21432
2    21720
3    22024
4    21994
Name: count, dtype: int64

(stock_code, year, quarter) 정렬 검증: True


In [ ]:
# preprocessing
# 1. object dtype → numeric 변환 (inf 포함 문자열 처리)
obj_num_cols = quarter_df.select_dtypes(include='object').columns.difference(['stock_code'])
for col in obj_num_cols:
    quarter_df[col] = pd.to_numeric(quarter_df[col], errors='coerce')

# 2. inf → NaN
quarter_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 4. 업종코드 병합 (impute_by_group에서 업종×연도 그룹화에 사용)
industry_df = pd.read_csv('../data/industry_codes.csv', dtype={'stock_code': str})
industry_df['stock_code'] = industry_df['stock_code'].str.zfill(6)
industry_df = industry_df[['stock_code', 'induty_code']].drop_duplicates(subset='stock_code')
quarter_df = quarter_df.merge(industry_df, on='stock_code', how='left')
print(f"업종코드 매핑: {quarter_df['induty_code'].notna().sum():,}건 / {len(quarter_df):,}건")

업종코드 매핑: 73,113건 / 87,170건


/tmp/ipykernel_268070/3928897353.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_num_cols = quarter_df.select_dtypes(include='object').columns.difference(['stock_code'])


In [26]:
# 분기별 결측 비율 탐색 
exclude_cols = {'stock_code', 'year', 'quarter', 'induty_code'}
num_cols = [c for c in quarter_df.columns if c not in exclude_cols]

miss_by_quarter = (
    quarter_df.groupby('quarter')[num_cols]
    .apply(lambda g: g.isna().mean() * 100)
    .T
    .rename(columns={1: 'Q1(%)', 2: 'Q2(%)', 3: 'Q3(%)', 4: 'Q4(%)'})
    .round(1)
)

# 연간 모델(Q4): Q4 결측만 기준
def verdict_annual(row):
    q4 = row['Q4(%)']
    if q4 >= 20: return 'DROP'
    return 'OK'

# 분기 모델(Q1~Q4): 전 분기 max 기준
def verdict_quarterly(row):
    m = max(row['Q1(%)'], row['Q2(%)'], row['Q3(%)'], row['Q4(%)'])
    if m >= 20: return 'DROP'
    return 'OK'

miss_by_quarter['annual_verdict']    = miss_by_quarter.apply(verdict_annual, axis=1)
miss_by_quarter['quarterly_verdict'] = miss_by_quarter.apply(verdict_quarterly, axis=1)
miss_by_quarter = miss_by_quarter.sort_values(['quarterly_verdict', 'Q4(%)'], ascending=[True, False])

print("=== 분기별 결측 비율 (%) ===")
print(miss_by_quarter.to_string())

print("\n[연간 모델 verdict (Q4 기준)]")
print(miss_by_quarter['annual_verdict'].value_counts())
print("\n[분기 모델 verdict (전분기 max 기준)]")
print(miss_by_quarter['quarterly_verdict'].value_counts())

# 모델별 사용 가능 피처 목록 출력
annual_ok    = miss_by_quarter[miss_by_quarter['annual_verdict']    != 'DROP'].index.tolist()
quarterly_ok = miss_by_quarter[miss_by_quarter['quarterly_verdict'] != 'DROP'].index.tolist()

ANNUAL_FEATURES = annual_ok
QUARTERLY_FEATURES = quarterly_ok

print(f"\n연간 모델 사용 가능 피처 ({len(annual_ok)}개):    {annual_ok}")
print(f"\n분기 모델 사용 가능 피처 ({len(quarterly_ok)}개): {quarterly_ok}")

=== 분기별 결측 비율 (%) ===
quarter                      Q1(%)  Q2(%)  Q3(%)  Q4(%) annual_verdict quarterly_verdict
sales_growth_rate             46.8   46.6   13.8    8.9             OK              DROP
net_income_growth             68.5   70.0   28.6    5.8             OK              DROP
op_income_growth              70.5   69.7   17.5    5.7             OK              DROP
net_margin                    89.0   90.5    2.7    4.5             OK              DROP
op_margin                     84.5   84.2    4.2    4.3             OK              DROP
asset_turnover_ratio          53.8   53.3    0.5    3.0             OK              DROP
retained_earnings             69.8   68.7   71.6    0.9             OK              DROP
retained_earnings_ratio       70.1   69.0   71.9    0.9             OK              DROP
retained_earnings_to_equity   71.2   70.5   73.1    0.9             OK              DROP
roa                           83.9   86.1    2.6    0.3             OK              DROP

In [ ]:
DROP_THRESHOLD     = 0.20   # 컬럼 단위: 20% 이상 → 제거
ROW_DROP_THRESHOLD = 0.50   # 행 단위: 피처 중 50% 이상 결측인 행 제거

META_COLS = ['stock_code', 'year', 'quarter', 'induty_code']

def preprocess_by_threshold(df, feature_cols, growth_cols=None, label=''):
    """
    전처리 순서:
     1. [행 단위] 피처 50% 초과 결측 행 제거
     2. [컬럼 단위] 20% 초과 결측 컬럼 제거
     3. [성장률] 퍼센타일 윈저라이제이션
     4. [행 단위] 시계열 정렬 후 동일 기업 직전값 ffill
     5. [행 단위] ffill 후에도 결측 잔존 행 제거
     6. [컬럼 선택] META_COLS + 생존 피처만 반환
    """
    df = df.copy()
    dropped = []

    # ── 1. 행 단위: 결측 과다 행 제거 ────────────────────────
    existing_feat = [c for c in feature_cols if c in df.columns]
    row_miss = df[existing_feat].isna().mean(axis=1)
    n_row_drop = (row_miss > ROW_DROP_THRESHOLD).sum()
    if n_row_drop > 0:
        df = df[row_miss <= ROW_DROP_THRESHOLD].copy()
        print(f"  [행 제거 1] 피처 결측 >{ROW_DROP_THRESHOLD*100:.0f}%: {n_row_drop}행 → 남은 행: {len(df):,}")

    # ── 2. 컬럼 단위: 결측 과다 컬럼 제거 ───────────────────
    total = len(df)
    for col in list(feature_cols):
        if col not in df.columns:
            continue
        pct = df[col].isna().sum() / total
        if pct >= DROP_THRESHOLD:
            df.drop(columns=[col], inplace=True)
            dropped.append(f'{col} ({pct*100:.1f}%)')

    # ── 3. 성장률 이상치 윈저라이제이션 ──────────────────────
    for col in (growth_cols or []):
        if col in df.columns:
            lo, hi = df[col].quantile([0.01, 0.99])
            df[col] = df[col].clip(lo, hi)

    # ── 4. 시계열 정렬 후 ffill ──────────────────────────────
    # 반드시 정렬 후 ffill — 미정렬 시 미래값이 과거로 채워지는 누수 발생
    sort_keys = [k for k in ['stock_code', 'year', 'quarter'] if k in df.columns]
    df = df.sort_values(sort_keys).reset_index(drop=True)

    surviving = [c for c in feature_cols if c in df.columns]
    if 'stock_code' in df.columns:
        df[surviving] = df.groupby('stock_code')[surviving].transform(lambda s: s.ffill())

    # ── 5. 행 단위: ffill 후 잔존 결측 행 제거 ───────────────
    remaining_mask = df[surviving].isna().any(axis=1)
    n_remaining_drop = remaining_mask.sum()
    if n_remaining_drop > 0:
        df = df[~remaining_mask].copy()
        print(f"  [행 제거 2] ffill 후 결측 잔존: {n_remaining_drop}행 → 남은 행: {len(df):,}")

    remaining = df[surviving].isnull().sum().sum()
    print(f"[{label}] 컬럼 DROP={len(dropped)}  잔여결측={remaining}건")
    if dropped:
        print(f"  DROP: {dropped}")

    # ── 6. META_COLS + 생존 피처만 반환 ──────────────────────
    keep_cols = [c for c in META_COLS if c in df.columns] + surviving
    df = df[keep_cols]
    print(f"  → 최종 shape: {df.shape}  (피처: {len(surviving)}개)")
    return df

In [ ]:
annual_df = quarter_df[quarter_df['quarter'] == 4].copy()
print(f"Q4 필터링: {len(annual_df):,}행\n")

ANNUAL_GROWTH_COLS = ['sales_growth_rate', 'op_income_growth', 'net_income_growth', 'asset_growth']
annual_clean = preprocess_by_threshold(annual_df, ANNUAL_FEATURES, ANNUAL_GROWTH_COLS, '연간 (Q4)')

print(f"\n[연간 클린 컬럼 ({len(annual_clean.columns)}개)]")
print(f"  META : {[c for c in META_COLS if c in annual_clean.columns]}")
print(f"  피처 : {[c for c in annual_clean.columns if c not in META_COLS]}")

In [29]:
# ============================================================
# 분기 모델 전처리 (Q1~Q4 전체)
# ============================================================
quarterly_df = quarter_df.copy()
print(f"전체 분기 데이터: {len(quarterly_df):,}행\n")

# 전처리 적용
quarterly_clean = preprocess_by_threshold(quarterly_df, QUARTERLY_FEATURES, ['asset_growth'], '분기 (Q1~Q4)')

print("\n[분기별 행 수]")
print(quarterly_clean['quarter'].value_counts().sort_index())

전체 분기 데이터: 87,170행

  [행 제거 1] 피처 결측 >50%: 2038행 → 남은 행: 85,132
  [행 제거 2] ffill 후 결측 잔존: 10127행 → 남은 행: 75,005
[분기 (Q1~Q4)] 컬럼 DROP=0  잔여결측=0건
  → 최종 shape: (75005, 36)

[분기별 행 수]
quarter
1    17878
2    18227
3    18524
4    20376
Name: count, dtype: int64


### Score 계산

In [36]:
# ============================================================
# 라벨 데이터 로드 (연간/분기 공통)
# ============================================================

# 상장폐지
krx_delisting = pd.read_csv('../data/krx_delisting.csv')
krx_delisting['Symbol'] = krx_delisting['Symbol'].astype(str).str.zfill(6)
krx_delisting = krx_delisting[
    (krx_delisting['DelistingDate'] >= '2016-01-01') &
    (krx_delisting['DelistingDate'] <= '2025-12-31')
].copy()
krx_delisting['DelistingDate'] = pd.to_datetime(krx_delisting['DelistingDate'])
krx_delisting['delisting_year']    = krx_delisting['DelistingDate'].dt.year
krx_delisting['delisting_quarter'] = krx_delisting['DelistingDate'].dt.quarter

# 연간 모델용 (year only)
delisting_map = krx_delisting.groupby('Symbol')['delisting_year'].min().reset_index()
delisting_map.columns = ['stock_code', 'delisting_year']

# 분기 모델용 (year + quarter, 최초 상장폐지 날짜 기준)
delisting_map_q = (
    krx_delisting.sort_values('DelistingDate')
    .groupby('Symbol')[['delisting_year', 'delisting_quarter']]
    .first()
    .reset_index()
)
delisting_map_q.columns = ['stock_code', 'delisting_year', 'delisting_quarter']

# 관리종목
krx_adm = pd.read_csv('../data/krx_adm.csv')
krx_adm['Symbol'] = krx_adm['Symbol'].astype(str).str.zfill(6)
krx_adm = krx_adm[
    (krx_adm['DesignationDate'] >= '2016-01-01') &
    (krx_adm['DesignationDate'] <= '2025-12-31')
].copy()
krx_adm['DesignationDate'] = pd.to_datetime(krx_adm['DesignationDate'])
krx_adm['designation_year']    = krx_adm['DesignationDate'].dt.year
krx_adm['designation_quarter'] = krx_adm['DesignationDate'].dt.quarter

# 연간 모델용 (year only)
adm_map = krx_adm.groupby('Symbol')['designation_year'].min().reset_index()
adm_map.columns = ['stock_code', 'designation_year']

# 분기 모델용 (year + quarter, 최초 지정 날짜 기준)
adm_map_q = (
    krx_adm.sort_values('DesignationDate')
    .groupby('Symbol')[['designation_year', 'designation_quarter']]
    .first()
    .reset_index()
)
adm_map_q.columns = ['stock_code', 'designation_year', 'designation_quarter']

print(f"상장폐지 종목: {delisting_map['stock_code'].nunique()}개  ({delisting_map['delisting_year'].min()}~{delisting_map['delisting_year'].max()})")
print(f"관리종목:      {adm_map['stock_code'].nunique()}개  ({adm_map['designation_year'].min()}~{adm_map['designation_year'].max()})")
print(f"\n[분기 분포 - 상장폐지]")
print(delisting_map_q['delisting_quarter'].value_counts().sort_index())
print(f"\n[분기 분포 - 관리종목]")
print(adm_map_q['designation_quarter'].value_counts().sort_index())

상장폐지 종목: 1355개  (2016~2025)
관리종목:      84개  (2022~2025)

[분기 분포 - 상장폐지]
delisting_quarter
1    231
2    339
3    372
4    413
Name: count, dtype: int64

[분기 분포 - 관리종목]
designation_quarter
1    34
2    30
3    11
4     9
Name: count, dtype: int64


In [31]:
# ============================================================
# 연간 데이터 라벨링 + 파생변수
# ============================================================
annual_labeled = annual_clean.copy()

# ── 상장폐지 ───────────────────────────────────────────────
annual_labeled = annual_labeled.merge(delisting_map, on='stock_code', how='left')

# 상장폐지 연도 이후 데이터 제거 (당해연도 유지)
annual_labeled = annual_labeled[
    annual_labeled['delisting_year'].isna() |
    (annual_labeled['year'] <= annual_labeled['delisting_year'])
].copy()

# delisted=1: 종목별 마지막 year 행 하나만
annual_labeled['delisted'] = 0
last_year_idx = (
    annual_labeled[annual_labeled['delisting_year'].notna()]
    .groupby('stock_code')['year']
    .idxmax()
)
annual_labeled.loc[last_year_idx, 'delisted'] = 1
annual_labeled.drop(columns=['delisting_year'], inplace=True)

# ── 관리종목 ───────────────────────────────────────────────
annual_labeled = annual_labeled.merge(adm_map, on='stock_code', how='left')

annual_labeled['admin_designated'] = 0
mask = (
    annual_labeled['designation_year'].notna() &
    (annual_labeled['year'] >= annual_labeled['designation_year'])
)
annual_labeled.loc[mask, 'admin_designated'] = 1
annual_labeled.drop(columns=['designation_year'], inplace=True)


# ── 결과 요약 ───────────────────────────────────────────────
print(f"연간 데이터: {len(annual_labeled):,}행  {annual_labeled.shape[1]}열")
print(f"\n[delisted 분포]")
print(annual_labeled['delisted'].value_counts())
print(f"  상장폐지 종목 수: {annual_labeled[annual_labeled['delisted']==1]['stock_code'].nunique()}개")
print(f"\n[admin_designated 분포]")
print(annual_labeled['admin_designated'].value_counts())
print(f"  관리종목 종목 수: {annual_labeled[annual_labeled['admin_designated']==1]['stock_code'].nunique()}개")

연간 데이터: 18,983행  38열

[delisted 분포]
delisted
0    18844
1      139
Name: count, dtype: int64
  상장폐지 종목 수: 139개

[admin_designated 분포]
admin_designated
0    18872
1      111
Name: count, dtype: int64
  관리종목 종목 수: 79개


In [37]:
# ============================================================
# 분기 데이터 라벨링
# ============================================================
quarterly_labeled = quarterly_clean.copy()

# yq: (year, quarter) → 정렬 가능한 정수 (비교 연산에 사용)
quarterly_labeled['yq'] = quarterly_labeled['year'] * 4 + quarterly_labeled['quarter']

# ── 상장폐지 ───────────────────────────────────────────────
quarterly_labeled = quarterly_labeled.merge(delisting_map_q, on='stock_code', how='left')
quarterly_labeled['yq_delist'] = (
    quarterly_labeled['delisting_year'] * 4 + quarterly_labeled['delisting_quarter']
)

# 상장폐지 분기 이후 데이터 제거 (당해 분기 포함 유지)
quarterly_labeled = quarterly_labeled[
    quarterly_labeled['yq_delist'].isna() |
    (quarterly_labeled['yq'] <= quarterly_labeled['yq_delist'])
].copy()

# delisted=1 마킹
quarterly_labeled['delisted'] = 0
delist_df = quarterly_labeled[quarterly_labeled['yq_delist'].notna()].copy()

# 정확한 상장폐지 분기 데이터가 있는 경우 → 해당 분기 행에 마킹
exact_mask = (delist_df['yq'] == delist_df['yq_delist'])
quarterly_labeled.loc[delist_df[exact_mask].index, 'delisted'] = 1

# 정확한 분기 데이터가 없는 종목 → 직전 마지막 가용 분기에 마킹 (fallback)
exact_codes = delist_df.loc[exact_mask, 'stock_code'].unique()
fallback_df = delist_df[
    ~delist_df['stock_code'].isin(exact_codes) &
    (delist_df['yq'] <= delist_df['yq_delist'])
]
fallback_idx = fallback_df.groupby('stock_code')['yq'].idxmax()
quarterly_labeled.loc[fallback_idx, 'delisted'] = 1

quarterly_labeled.drop(columns=['delisting_year', 'delisting_quarter', 'yq_delist'], inplace=True)

# ── 관리종목 ───────────────────────────────────────────────
quarterly_labeled = quarterly_labeled.merge(adm_map_q, on='stock_code', how='left')
quarterly_labeled['yq_adm'] = (
    quarterly_labeled['designation_year'] * 4 + quarterly_labeled['designation_quarter']
)

# admin_designated=1: 지정 분기 이후 전 행
quarterly_labeled['admin_designated'] = 0
mask = (
    quarterly_labeled['yq_adm'].notna() &
    (quarterly_labeled['yq'] >= quarterly_labeled['yq_adm'])
)
quarterly_labeled.loc[mask, 'admin_designated'] = 1
quarterly_labeled.drop(columns=['designation_year', 'designation_quarter', 'yq_adm'], inplace=True)

# ── 결과 요약 ───────────────────────────────────────────────
print(f"분기 데이터: {len(quarterly_labeled):,}행  {quarterly_labeled.shape[1]}열")
print(f"\n[delisted 분포]")
print(quarterly_labeled['delisted'].value_counts())
print(f"  상장폐지 종목 수: {quarterly_labeled[quarterly_labeled['delisted']==1]['stock_code'].nunique()}개")
print(f"\n  [delisted=1 분기 분포]")
print(quarterly_labeled[quarterly_labeled['delisted']==1]['quarter'].value_counts().sort_index())
print(f"\n[admin_designated 분포]")
print(quarterly_labeled['admin_designated'].value_counts())
print(f"  관리종목 종목 수: {quarterly_labeled[quarterly_labeled['admin_designated']==1]['stock_code'].nunique()}개")
print(f"\n[분기별 행 수]")
print(quarterly_labeled['quarter'].value_counts().sort_index())

분기 데이터: 74,378행  39열

[delisted 분포]
delisted
0    74143
1      235
Name: count, dtype: int64
  상장폐지 종목 수: 235개

  [delisted=1 분기 분포]
quarter
1     24
2     46
3     58
4    107
Name: count, dtype: int64

[admin_designated 분포]
admin_designated
0    74037
1      341
Name: count, dtype: int64
  관리종목 종목 수: 79개

[분기별 행 수]
quarter
1    17843
2    18067
3    18332
4    20136
Name: count, dtype: int64


In [ ]:
# 감사의견 처리
import re

audit_df = pd.read_csv('../data/audit_all.csv')
audit_df['stock_code'] = audit_df['stock_code'].str.zfill(6)

# stlm_dt → 연도/분기 추출
audit_df['stlm_dt'] = pd.to_datetime(audit_df['stlm_dt'], errors='coerce')
audit_df = audit_df.dropna(subset=['stlm_dt'])   # stlm_dt 없는 행 제거
audit_df['stlm_year']    = audit_df['stlm_dt'].dt.year
audit_df['stlm_quarter'] = audit_df['stlm_dt'].dt.quarter

def normalize_opinion(x):
    s = str(x).strip().lower().replace("\n", " ")
    s = re.sub(r"[^가-힣a-z0-9]", "", s)
    return s

def map_opinion(raw):
    s = normalize_opinion(raw)
    if re.search(r"의견거절|거절", s):
        return "의견거절"
    if "부적정" in s:
        return "부적정"
    if "한정" in s:
        return "한정"
    if re.search(r"(?<!부)적정|적정의견|공정", s):
        return "적정"
    return "기타/확인필요"

audit_df["audit_state"] = audit_df["audit_opinion"].apply(map_opinion)

print(audit_df['audit_state'].value_counts())
print(f'\n[stlm_quarter 분포]')
print(audit_df['stlm_quarter'].value_counts().sort_index())

In [ ]:
audit_df

In [ ]:
penalty_map = {'적정': 0, '한정': 30, '부적정': 60, '의견거절': 60, '기타/확인필요': 0}
audit_df['audit_penalty'] = audit_df['audit_state'].map(penalty_map).fillna(0)

priority = ['의견거절', '부적정', '한정', '적정', '기타/확인필요']

def pick_worst_state(states):
    st = set(states.dropna())
    for p in priority:
        if p in st: return p
    return '기타/확인필요'

# 당기(현재 기간) 감사의견만 사용 — 전기/전전기 중복 제외
audit_current = audit_df[audit_df['bsns_year'].str.contains('당기', na=False)].copy()

# stlm_dt 기준 (stock_code, 연도, 분기)별 최악 의견 집계
quarter_penalty_df = (
    audit_current
    .groupby(['stock_code', 'stlm_year', 'stlm_quarter'], as_index=False)
    .agg(
        audit_state_final=('audit_state', pick_worst_state),
        audit_penalty_final=('audit_penalty', 'max'),
    )
    .rename(columns={'stlm_year': 'year', 'stlm_quarter': 'quarter'})
)

print(f"분기별 감사의견 데이터: {len(quarter_penalty_df):,}건")
print(quarter_penalty_df['audit_state_final'].value_counts())
print(f'\n[분기 분포]')
print(quarter_penalty_df['quarter'].value_counts().sort_index())

In [ ]:
# ============================================================
# 계단식 패널티 함수
# ============================================================
def stepped_penalty_from_future(df, flag_col, base_penalty, horizon=3, step=10):
    g = df.groupby('stock_code')[flag_col]
    cands = []
    for k in range(horizon + 1):
        w = base_penalty - step * k
        if w <= 0: continue
        cands.append(g.shift(-k).fillna(0).astype(int) * w)
    return pd.concat(cands, axis=1).max(axis=1)

# ============================================================
# 연간 Risk Score (t+1 타겟)
# 연간 감사의견: stlm_dt Q4 (12월 결산) 기준
# ============================================================
annual_audit = (
    quarter_penalty_df[quarter_penalty_df['quarter'] == 4]
    [['stock_code', 'year', 'audit_penalty_final']]
)

annual_scored = (
    annual_labeled
    .merge(annual_audit, on=['stock_code', 'year'], how='left')
    .sort_values(['stock_code', 'year'])
    .reset_index(drop=True)
)
annual_scored['audit_penalty_final'] = annual_scored['audit_penalty_final'].fillna(0)

# t+1 라벨 (연 단위 shift)
annual_scored['delisted_t1']      = annual_scored.groupby('stock_code')['delisted'].shift(-1)
annual_scored['admin_t1']         = annual_scored.groupby('stock_code')['admin_designated'].shift(-1)
annual_scored['audit_penalty_t1'] = annual_scored.groupby('stock_code')['audit_penalty_final'].shift(-1).fillna(0)

# 계단식 패널티
annual_scored['delisted_penalty_t1'] = stepped_penalty_from_future(
    annual_scored, 'delisted_t1', base_penalty=40, horizon=3, step=10
)
annual_scored['admin_penalty_t1'] = stepped_penalty_from_future(
    annual_scored, 'admin_t1', base_penalty=30, horizon=2, step=10
)

# 최종 타겟 스코어
annual_scored['target_score_t1'] = (
    100
    - annual_scored['audit_penalty_t1']
    - annual_scored['delisted_penalty_t1']
    - annual_scored['admin_penalty_t1']
).clip(0, 100)

annual_model_df = annual_scored.dropna(subset=['target_score_t1']).copy()

print(f"연간 모델 데이터: {len(annual_model_df):,}행")
print(f"\n[target_score_t1 분포]")
print(annual_model_df['target_score_t1'].value_counts().sort_index().head(10))
print(f"\n고위험(≤70) 비율: {(annual_model_df['target_score_t1'] <= 70).mean()*100:.2f}%")

In [ ]:
# ============================================================
# 분기 Risk Score (t+1 분기 타겟)
# 감사의견: stlm_dt 기준 (stock_code, year, quarter) 직접 매칭
# ============================================================
quarterly_scored = (
    quarterly_labeled
    .merge(
        quarter_penalty_df[['stock_code', 'year', 'quarter',
                             'audit_penalty_final', 'audit_state_final']],
        on=['stock_code', 'year', 'quarter'], how='left'
    )
    .sort_values(['stock_code', 'year', 'quarter'])
    .reset_index(drop=True)
)
quarterly_scored['audit_penalty_final'] = quarterly_scored['audit_penalty_final'].fillna(0)

# t+1 분기 라벨
quarterly_scored['delisted_t1'] = quarterly_scored.groupby('stock_code')['delisted'].shift(-1)
quarterly_scored['admin_t1']    = quarterly_scored.groupby('stock_code')['admin_designated'].shift(-1)

# 감사의견 t+1: 다음 분기 shift (-1) — 연간/반기 모두 분기 단위로 이동
quarterly_scored['audit_penalty_t1'] = (
    quarterly_scored.groupby('stock_code')['audit_penalty_final']
    .shift(-1)
    .fillna(0)
)

# 분기 계단식 패널티
quarterly_scored['delisted_penalty_t1'] = stepped_penalty_from_future(
    quarterly_scored, 'delisted_t1', base_penalty=40, horizon=3, step=10
)
quarterly_scored['admin_penalty_t1'] = stepped_penalty_from_future(
    quarterly_scored, 'admin_t1', base_penalty=30, horizon=2, step=10
)

# 최종 분기 타겟 스코어
quarterly_scored['target_score_t1'] = (
    100
    - quarterly_scored['audit_penalty_t1']
    - quarterly_scored['delisted_penalty_t1']
    - quarterly_scored['admin_penalty_t1']
).clip(0, 100)

quarterly_model_df = quarterly_scored[quarterly_scored['delisted_t1'].notna()].copy()
quarterly_model_df = quarterly_model_df.drop(columns=['yq'], errors='ignore')

print(f"분기 모델 데이터: {len(quarterly_model_df):,}행")
print(f"\n[분기별 행 수]")
print(quarterly_model_df['quarter'].value_counts().sort_index())
print(f"\n[target_score_t1 분포]")
print(quarterly_model_df['target_score_t1'].value_counts().sort_index().head(10))
print(f"\n고위험(≤70) 비율: {(quarterly_model_df['target_score_t1'] <= 70).mean()*100:.2f}%")
print(f"\n[패널티 발생 건수]")
print(f"  delisted_penalty_t1 > 0: {(quarterly_model_df['delisted_penalty_t1'] > 0).sum()}")
print(f"  admin_penalty_t1 > 0:    {(quarterly_model_df['admin_penalty_t1'] > 0).sum()}")
print(f"  audit_penalty_t1 > 0:    {(quarterly_model_df['audit_penalty_t1'] > 0).sum()}")

In [43]:
# 최종 데이터
annual_model_df.to_csv('../data/annual_model.csv', index=False)
quarterly_model_df.to_csv('../data/quarterly_model.csv', index=False)